<a href="https://colab.research.google.com/github/toche7/SlideAIDATADGA/blob/main/Workshop3v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop 3: Data Preparation Pipeline

In [ ]:
# ดาวน์โหลด TH Sarabun New
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

# 1) Import libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# เพิ่มฟอนต์เข้าระบบ Matplotlib
fm.fontManager.addfont('thsarabunnew-webfont.ttf')
# ตั้งค่า default font ให้เป็น TH Sarabun New
mpl.rc('font', family='TH Sarabun New')
plt.rcParams.update({'font.size': 14})

# ทดสอบการพล็อตภาษาไทย
plt.figure(figsize=(6, 4))
plt.text(0.5, 0.5, 'สวัสดีภาษาไทย (Thai Language Test)', fontsize=25, ha='center', va='center')
# plt.title('ทดสอบการแสดงผลภาษาไทย')
plt.axis('off')
plt.show()

## 1. การโหลดข้อมูล

ขั้นตอนนี้เราจะโหลดไฟล์ CSV ที่มีชื่อ `/content/traffic_accident.csv` เข้ามาใน DataFrame ของ Pandas เพื่อเริ่มต้นการวิเคราะห์ข้อมูล
เนื่องจากข้อมูลมีภาษาไทยและเกิดข้อผิดพลาดในการถอดรหัส (UnicodeDecodeError) เราจึงระบุ `encoding='TIS-620'` เพื่อให้สามารถอ่านไฟล์ได้อย่างถูกต้อง

In [ ]:
url = "https://github.com/toche7/DataSets/raw/refs/heads/main/traffic_accident.csv"

In [ ]:
df = pd.read_csv(url, encoding='TIS-620')
display(df.head())

## 2. ตรวจสอบคุณภาพข้อมูลเบื้องต้น (ค่าที่ขาดหายไป)

หลังจากโหลดข้อมูลแล้ว เราจะตรวจสอบคุณภาพของข้อมูลโดยดูว่ามีค่าที่ขาดหายไป (Missing Values) อยู่ในคอลัมน์ใดบ้าง และคิดเป็นกี่เปอร์เซ็นต์ของข้อมูลทั้งหมด เพื่อประเมินผลกระทบที่อาจเกิดขึ้นต่อการวิเคราะห์

In [ ]:
print('DataFrame Info:')
df.info()

In [ ]:
print('\nMissing values percentage per column:')
missing_percentage = df.isnull().sum() / len(df) * 100
display(missing_percentage[missing_percentage > 0].sort_values(ascending=False))

## 3. การจัดการกับค่าที่ขาดหายไป

จากผลการตรวจสอบ เราพบว่ามีค่าที่ขาดหายไปเป็นจำนวนน้อยมาก (ต่ำกว่า 2%) ในหลายๆ คอลัมน์ เพื่อให้ข้อมูลสะอาดและพร้อมสำหรับการวิเคราะห์ในขั้นต่อไป เราจึงเลือกที่จะลบแถวข้อมูลที่มีค่าที่ขาดหายไปออกทั้งหมด และสร้างเป็น DataFrame ใหม่ชื่อ `df_cleaned` ซึ่งเป็นสำเนาที่สมบูรณ์ (deep copy) เพื่อหลีกเลี่ยงข้อผิดพลาด `SettingWithCopyWarning`

In [ ]:
# Drop rows with any missing values
df_cleaned = df.dropna().copy()

print(f"Original DataFrame shape: {df.shape}")
print(f"Cleaned DataFrame shape: {df_cleaned.shape}")

## 4. ตรวจสอบค่าที่ขาดหายไปหลังการทำความสะอาด

หลังจากลบแถวที่มีค่าที่ขาดหายไปออกไปแล้ว เราจะตรวจสอบอีกครั้งเพื่อยืนยันว่าไม่มีค่าที่ขาดหายไปเหลืออยู่ใน `df_cleaned` แล้ว เพื่อให้มั่นใจว่าข้อมูลสะอาดจริง

In [ ]:
# Verify that there are no more missing values
print('\nMissing values percentage after dropping rows:')
missing_percentage_cleaned = df_cleaned.isnull().sum() / len(df_cleaned) * 100
display(missing_percentage_cleaned[missing_percentage_cleaned > 0].sort_values(ascending=False))

## 5. ตรวจสอบค่าที่ไม่ซ้ำกันของแต่ละคอลัมน์

ขั้นตอนนี้เป็นการสำรวจข้อมูลเชิงลึกมากขึ้น โดยการแสดงค่าที่ไม่ซ้ำกันทั้งหมดในแต่ละคอลัมน์ของ `df_cleaned` เพื่อทำความเข้าใจประเภทของข้อมูลและความหลากหลายของข้อมูลในแต่ละแอตทริบิวต์ ช่วยให้เรามองเห็นรูปแบบหรือค่าผิดปกติได้ง่ายขึ้น

In [ ]:
print("Unique values in 'Days of a week' column:")
display(df_cleaned['Days of a week'].unique())

In [ ]:
for column in df_cleaned.columns:
    print(f"\nUnique values in '{column}' column:")
    print(df_cleaned[column].unique())

## 6. การปรับปรุงข้อมูล: ลบช่องว่างนำหน้า/ท้ายในคอลัมน์ 'road/bridge'

ตามคำแนะนำของคุณผู้ใช้ เราพบว่าคอลัมน์ 'road/bridge' อาจมีช่องว่างที่ไม่จำเป็นทั้งก่อนและหลังข้อความ ซึ่งอาจส่งผลต่อการวิเคราะห์หรือการจัดกลุ่มข้อมูล เพื่อแก้ไขปัญหานี้ เราจะใช้เมธอด `.str.strip()` เพื่อลบช่องว่างเหล่านั้นออก ทำให้ข้อมูลมีความสม่ำเสมอมากขึ้น และแสดงค่าที่ไม่ซ้ำกันอีกครั้งเพื่อยืนยันการเปลี่ยนแปลง

In [ ]:
df_cleaned['road/bridge'] = df_cleaned['road/bridge'].str.strip()
print("Unique values in 'road/bridge' column after trimming spaces:")
print(df_cleaned['road/bridge'].unique())

In [ ]:
df_cleaned.to_csv('output.csv', index=False, encoding='TIS-620')
print("DataFrame 'df_cleaned' saved to 'output.csv' successfully.")